# Práctica Analisis Texto lo que sea

Explicar aquí

In [81]:
#Imports:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import os
import json

%matplotlib inline

## Paso 1: Extraer todos los datos 

Aquí básicamente vamos a tener que extraer todos los datos y meterlos en un dataset. Podemos usar pandas para eso. Además tendremos que limpiar los datos y 'organizarlos'. Podemos hacer una lista de listas (es decir, una lista con todos los jugadores/sesiones y cada uno de ellos con una lista de todos sus eventos relaccionados. si nos aseguramos que cada documento es TODA la lista de eventos del jugador, este paso será bastante fácil.)

In [82]:
path_to_json_files = '../Assets/Sessions/' 

#metodo que coge todos los archivos json en un directorio
json_file_names = [filename for filename in os.listdir(path_to_json_files) if filename.endswith('.json')]
#print(json_file_names)
jsonList=[0]*len(json_file_names)

for i in range(len(json_file_names)):
    jsonList[i]=pd.read_json(path_to_json_files+json_file_names[i])

print(jsonList[0])
print(jsonList[0].columns)


          type    time                sesID  levelID  checkpointID  cordX  \
0     sesStart   12881  1190520571561792703      NaN           NaN    NaN   
1     playerCP   12881  1190520571561792703      2.0       -4802.0    NaN   
2     playShot   21194  1190520571561792703      NaN           NaN    NaN   
3     playShot   21336  1190520571561792703      NaN           NaN    NaN   
4     playShot   21479  1190520571561792703      NaN           NaN    NaN   
..         ...     ...                  ...      ...           ...    ...   
218   playShot  151829  1190520571561792703      NaN           NaN    NaN   
219   playShot  153798  1190520571561792703      NaN           NaN    NaN   
220  playerEnd  155792  1190520571561792703      2.0           NaN    NaN   
221   playerCP  167552  1190520571561792703      3.0      -11596.0    NaN   
222     sesEnd  181884  1190520571561792703      NaN           NaN    NaN   

     cordY  hitCause  hitDamage  currentHealth  previousHealth  healingAmou

## Paso 2: Calcular las métricas individuales

Como muchas de las metricas no se calculan con la raw data de todos los jugadores, tendremos que extraer primero los datos necesarios.

In [83]:
#%%writefile filespy/necesaryData.py

# Creacion de las columnas que vamos a usar
col_names =  ['ID','MuTut', 'MuN1', 'MuN2', 'MuN3',  
              'MuSpike', 'MuE1', 'MuE2', 'MuE3', 'MuFi', 'MuIc',
              'Dis', 'DisAc', 'Mel', 'MelAc',
              'DamTut', 'DamN1', 'DamN2', 'DamN3',
              'DamSpike', 'DamE1', 'DamE2', 'DamE3', 'DamFi', 'DamIc',
              'Heal', 'OvHeal', 
              'ISesTime', 'SesTime', 'TimTut', 'TimN1', 'TimN2', 'TimN3']

# create an empty dataframe
# with columns
singleDf  = pd.DataFrame(columns = col_names)

# show the dataframe
singleDf

,ID,MuTut,MuN1,MuN2,MuN3,MuSpike,MuE1,MuE2,MuE3,MuFi,...,DamFi,DamIc,Heal,OvHeal,ISesTime,SesTime,TimTut,TimN1,TimN2,TimN3


In [84]:
#%%writefile filespy/jugadores.py
#En esta celda es donde haremos el for por cada jugador (sessionID) distinto.
#Si lo hacemos bien podemos usar las librerias para ahorrarnos el trabajo.

#jug = [[1, 2, 3], [4, 2], [3]]; #Valor de jugadores arbitrario
jug = jsonList
for i in range(len(jug)):
    dataJug = ['name' + str(i), 0, 0, 0, 0,
               0, 0, 0, 0, 0, 0, 
               0, 0, 0, 0,
               0, 0, 0, 0,
               0, 0, 0, 0, 0, 0,
               0, 0, 
               0, 0, 0, 0, 0, 0]
    

    
    #print(jug[i])
    #Loopear todos los eventos, y tener un switch para que segun event-type haga una cosa u otra.
    for j in range(len(jug[i])):
        #cogemos el tipo del evento para luego comparar en el switch
        event = jug[i].loc[j]["type"]
        #print(event)
        #print(aux)
        #match al type del json. 
        #Cambiar a event["type"] cuando sea un archivo json 
        match event:
            case "sesStart":
                dataJug[27] = jug[i].loc[j]["time"]
            case "playerEnd":
                indexOffset = int(jug[i].loc[j]["levelID"]) - 1 #indice del nivel que hemos acabado
                print(indexOffset)
                print(singleDf.columns.get_loc("TimTut"))

                dataJug[singleDf.columns.get_loc("TimTut") + indexOffset] = jug[i].loc[j]["time"]
            case "playerDeath":
                dataJug[4] += 1
                index = int(jug[i].loc[j]["levelID"] - 1) #indice del nivel que hemos acabado
                dataJug[jug[i].loc[j]["levelID"] - 1] += 1
                dataJug[jug[i].loc[j]["hitCause"] + 5] += 1
                #Poner las coordenadas de la muerte en una lista
            case "playShot": 
                dataJug[singleDf.columns.get_loc("Dis")] += 1
            case "enBulHit":
                dataJug[12] += 1
            case "playerMelee": 
                dataJug[13] += 1
            case "enemyMeleeHit":
                dataJug[14] += 1
            case "playerHit":
                dataJug[13] += jug[i].loc[j]["hitDamage"]
                dataJug[19] += jug[i].loc[j]["hitDamage"]
                #Poner las coordenadas de la muerte en una lista
            case "playerHeal":
                dataJug[25] += 1
                dataJug[26] += jug[i].loc[j]["previousHealth"] +jug[i].loc[j]["healingAmount"] - jug[i].loc[j]["finalHealth"]
            case "playerCP":
                #algo
                valor = 0
                #break
            case "sesEnd":
                dataJug[27] = jug[i].loc[j]["time"] - dataJug[27]
        #algomas
        
    #Posteriormente anexionamos jugoslavia (es broma solo anexionamos a tu madre)
    singleDf.loc[i] = dataJug

print(singleDf.iloc[0,11])#para ver valor del disparo del Jug0
singleDf


1
29
183


,ID,MuTut,MuN1,MuN2,MuN3,MuSpike,MuE1,MuE2,MuE3,MuFi,...,DamFi,DamIc,Heal,OvHeal,ISesTime,SesTime,TimTut,TimN1,TimN2,TimN3
0,name0,0,0,0,0,0,0,0,0,0,...,0,0,1,0.0,169003,0,0,155792,0,0


Aqui explicamos lo que hemos hecho.

## Paso 3: Calcular las métricas globales y Analisis

Aquí simplemente cogeríamos el dataset de los jugadores y, usando numpy y graficos super guapos de scipy y seaborn enseñar y analizarlos